# Post Image Processing for Ultrasound Simulation

**Student:** Darshil Maniya &nbsp;|&nbsp; **Roll No.:** 262SP009
**Programme:** M.Tech, Signal Processing and Machine Learning
**Department:** Electronics and Communication Engineering, NITK Surathkal
**Subject:** Seminar (EC787)

---

This notebook is the complete, self-contained implementation of the **Post Image
Processing** module of the NITK-UsoundSim project. The module is the final stage of
the 8-module simulated ultrasound scanner: it receives the raw, speckled B-mode
image from the upstream *B-Mode Image Formation* module and returns a
display-ready, despeckled image.

Because the upstream module had not yet delivered production output, a **synthetic
B-mode testbench** with known ground truth is built here (Section 3) and the
pipeline is developed and validated against it.

**Pipeline:** depth gain normalisation &rarr; despeckle filtering &rarr; contrast and
edge restoration &rarr; quality assessment.

All filter parameters are those documented in the project report (Table 4.2), and
the random seed is fixed at `7`, so the metrics computed in Section 8 reproduce the
report's Table 3.1 exactly. Section 9 then demonstrates the same pipeline on four
real clinical scans (liver, kidney, breast, thyroid) as a qualitative check.

> **Data dependency:** Section 9 reads four openly licensed images from `img/real/`
> (with `ATTRIBUTION.json`). Every other section is fully self-contained — if those
> files are absent, Sections 1–8 still run end to end.

## 1. Setup

Classical image processing only — no machine learning is used anywhere in this
module, which is a stated scope boundary of the project. The stack is NumPy/SciPy
for the array and filtering work, OpenCV for CLAHE and the Laplacian operator,
scikit-image for non-local means, wavelet thresholding and the reference metrics,
and PyWavelets (via scikit-image) for the Daubechies-4 basis.

In [ ]:
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import (gaussian_filter, gaussian_filter1d, median_filter,
                            uniform_filter, map_coordinates)
from skimage.restoration import denoise_nl_means, denoise_wavelet
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim

%matplotlib inline
plt.rcParams["image.cmap"] = "gray"
plt.rcParams["figure.dpi"] = 110

SEED = 7  # fixed so every number in this notebook is reproducible


def show_pair(before, after, title_after, title_before="BEFORE - raw"):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, im, t in zip(axes, [before, after], [title_before, title_after]):
        ax.imshow(im, vmin=0, vmax=255)
        ax.set_title(t, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print("numpy", np.__version__, "| opencv", cv2.__version__, "| pandas", pd.__version__)

## 2. Interface contract

A precise interface was agreed with the upstream B-Mode Image Formation team so the
two modules could be developed independently (report Table 4.1):

| Property | Input (received) | Output (returned) |
|---|---|---|
| Container | NumPy `.npy` array | NumPy `.npy` array |
| Data type | `uint8`, 2-D | `uint8`, 2-D |
| Array shape | $H \times W$ (nominally $560 \times 880$) | identical to input |
| Value range | 0–255, log-compressed | 0–255, display-ready |
| Geometry | Cartesian, sector (fan) format | identical geometry |
| Auxiliary | `mask` — boolean array marking valid fan pixels | — |

The module exposes a single public entry point:

```python
processed = post_process(raw_image, mask=mask, method="srad")
```

The `mask` argument is **required**: the fan-shaped valid region is surrounded by
zero-valued padding, and neighbourhood operators must not average genuine data with
that padding, which would introduce a dark halo along the sector boundary.

The five despeckle filters register themselves into `FILTERS` in Section 5, so the
dispatch table below is populated by the time `post_process` is first called.

In [ ]:
FILTERS = {}    # method name -> filter function, populated in Section 5
outputs = {}    # method name -> processed image, filled in as each stage runs


def despeckle(name):
    """Register a filter under its public `method` name."""
    def wrap(fn):
        FILTERS[name] = fn
        return fn
    return wrap


def to_uint8(x, mask=None):
    """Clip to display range and blank everything outside the valid fan."""
    out = np.clip(x, 0, 255).astype(np.uint8)
    if mask is not None:
        out = np.where(mask, out, 0).astype(np.uint8)
    return out


def post_process(raw_image, mask=None, method="srad", enhance=False):
    """Public entry point of the Post Image Processing module.

    raw_image : uint8 H x W log-compressed sector image from the upstream module
    mask      : boolean H x W array marking valid (in-fan) pixels
    method    : one of FILTERS  ('median', 'lee', 'srad', 'nlm', 'wavelet')
    enhance   : if True, apply the Stage 3 CLAHE + unsharp restoration
    """
    if method not in FILTERS:
        raise ValueError(f"unknown method {method!r}; available: {sorted(FILTERS)}")
    out = to_uint8(FILTERS[method](raw_image), mask)
    if enhance:
        out = to_uint8(clahe_unsharp(out), mask)
    return out


print("interface ready - filters register in Section 5")

## 3. Synthetic B-mode testbench

The testbench models the physics that produces a B-mode image, so that a *known*
ground truth exists against which the filters can be measured.

**Phantom.** A tissue reflectivity map $T(r,\theta)$ is defined in polar
(depth $\times$ scan-line) space containing a bright near-field skin layer, gentle
layered tissue banding, an anechoic cyst, a hyperechoic lesion, a small
low-contrast target, a hypoechoic nodule, and three point scatterers for resolution.

**Speckle synthesis.** Speckle arises from coherent summation of echoes from
sub-resolution scatterers. It is generated as a circular complex Gaussian field
scaled by $\sqrt{T}$ and convolved with an anisotropic point spread function, after
which the envelope is detected:

$$ e(r,\theta) \;=\; \bigl| \, h_{\text{PSF}} * \bigl( (n_r + j\,n_i)\sqrt{T} \bigr) \bigr| $$

Depth-dependent attenuation $e^{-\alpha r}$ and a small near-field clutter term are
then applied, followed by log compression to 8 bits over a 40 dB dynamic range, and
finally **scan conversion** from polar to the Cartesian fan geometry a clinical
scanner displays.

The ground-truth reference is built from the same $T$, PSF and attenuation but
**without** the random speckle field.

In [ ]:
NL, NS = 220, 520                             # scan lines (angle), samples (depth)
PSF_AX, PSF_LAT = 2.6, 3.4                     # PSF sigma: axial, lateral
ATTEN, DR = 1.15, 40.0                         # attenuation coefficient, dynamic range (dB)

th = np.linspace(-1, 1, NL)[None, :]           # normalised lateral
r = np.linspace(0, 1, NS)[:, None]             # normalised depth


def ellipse(cy, cx, ry, rx):
    return ((r - cy) / ry) ** 2 + ((th - cx) / rx) ** 2 <= 1.0


T = np.full((NS, NL), 1.0)
T += 0.35 * np.sin(r * 22.0) * 0.5              # layered tissue bands
T[r[:, 0] < 0.05, :] = 2.2                      # bright near-field skin layer
T[ellipse(0.42, -0.34, 0.13, 0.22)] = 0.04      # anechoic cyst
T[ellipse(0.60, 0.36, 0.12, 0.19)] = 4.5        # hyperechoic lesion
T[ellipse(0.28, 0.26, 0.05, 0.07)] = 5.5        # small low-contrast target
T[ellipse(0.80, -0.12, 0.07, 0.11)] = 0.12      # hypoechoic nodule
for (cy, cx) in [(0.16, -0.60), (0.16, 0.0), (0.16, 0.60)]:
    T[ellipse(cy, cx, 0.006, 0.010)] = 5.0      # point scatterers

print("reflectivity map:", T.shape, "| range", round(T.min(), 3), "-", round(T.max(), 1))

In [ ]:
def log_compress(env, dynamic_range=DR):
    """Envelope -> 8-bit log-compressed image over `dynamic_range` dB."""
    env = env / env.max()
    b = np.clip((20 * np.log10(env + 1e-4) + dynamic_range) / dynamic_range, 0, 1)
    return (b * 255).astype(np.uint8)


def scan_convert(p, H=560, W=880, ang=76.0, apex=0.28):
    """Polar (depth x angle) -> Cartesian fan image, plus validity mask."""
    NSr, NLr = p.shape
    half = np.radians(ang / 2.0)
    xmax = np.sin(half); ymin = apex * np.cos(half); ymax = 1.0
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float64)
    X = (xx / (W - 1)) * 2 * xmax - xmax
    Y = ymin + (yy / (H - 1)) * (ymax - ymin)
    rad = np.sqrt(X * X + Y * Y)
    ta = np.arctan2(X, Y)
    rr = (rad - apex) / (1.0 - apex) * (NSr - 1)
    cc = (ta / half + 1.0) / 2.0 * (NLr - 1)
    m = (rr >= 0) & (rr <= NSr - 1) & (np.abs(ta) <= half)
    cc = np.clip(cc, 0, NLr - 1)
    out = np.zeros((H, W))
    out[m] = map_coordinates(p.astype(np.float64), [rr[m], cc[m]], order=1)
    return out.astype(np.uint8), m


rng = np.random.default_rng(SEED)

field = (rng.normal(size=(NS, NL)) + 1j * rng.normal(size=(NS, NL))) * np.sqrt(T)
rf = (gaussian_filter(field.real, (PSF_AX, PSF_LAT))
      + 1j * gaussian_filter(field.imag, (PSF_AX, PSF_LAT)))
env = np.abs(rf)
env *= np.exp(-ATTEN * r)                                        # depth attenuation
env += 0.06 * np.abs(rng.normal(size=(NS, NL))) * np.exp(-6 * r)  # near-field clutter

raw, mask = scan_convert(log_compress(env))

amp = np.sqrt(gaussian_filter(T, (PSF_AX, PSF_LAT))) * np.exp(-ATTEN * r)
gt, _ = scan_convert(log_compress(amp))

print("raw:", raw.shape, raw.dtype, "| valid fan pixels:", int(mask.sum()))

In [ ]:
show_pair(raw, gt, "Ground truth (speckle-free reference)", "Raw B-mode (speckled)")

## 4. Stage 1 — Depth gain normalisation

Acoustic attenuation in soft tissue is approximately exponential with depth and
frequency, the received amplitude being scaled by $\exp(-\alpha f z)$ with
$\alpha \approx 0.5$ dB cm$^{-1}$ MHz$^{-1}$. Left uncorrected it darkens the lower
image and biases both the local-statistics filters and the contrast metrics.

Stage 1 estimates the mean intensity of each depth row *inside the valid mask* and
applies a smooth multiplicative correction so that row means follow a constant
target $\bar{I}_{\text{target}}$ (report Eq. 4.1):

$$ I_1(x,y) \;=\; I_0(x,y)\cdot\frac{\bar{I}_{\text{target}}}{\tilde{\mu}(y)+\varepsilon},
\qquad \tilde{\mu}(y) = \mathcal{G}_\sigma * \mu(y) $$

where $\mu(y)$ is the masked row mean, $\mathcal{G}_\sigma$ a one-dimensional
Gaussian kernel that prevents the correction from tracking anatomy rather than
attenuation, and $\varepsilon$ a small stabilising constant.

> **Note on the measured path.** The results reported in Section 8 — and in the
> report's Table 3.1 — are measured with the despeckle filters applied **directly to
> the raw scan-converted image**, matching the implementation the reported numbers
> were produced with. Stage 1 is implemented and demonstrated here for completeness
> of the pipeline; enabling it in the measured path would shift every metric and
> break parity with the report.

In [ ]:
def depth_gain_normalise(img, mask, sigma=25.0, eps=1e-6):
    """Flatten the depth-dependent brightness roll-off caused by attenuation."""
    x = img.astype(np.float64)
    valid = mask.astype(np.float64)
    row_cnt = valid.sum(axis=1)
    row_mean = np.divide((x * valid).sum(axis=1), row_cnt,
                          out=np.zeros(x.shape[0]), where=row_cnt > 0)
    smoothed = gaussian_filter1d(row_mean, sigma)
    target = row_mean[row_cnt > 0].mean()
    gain = target / (smoothed + eps)
    return to_uint8(x * gain[:, None], mask)


raw_dgc = depth_gain_normalise(raw, mask)
show_pair(raw, raw_dgc, "AFTER - depth gain normalised")

valid_rows = mask.sum(axis=1) > 0
prof = lambda im: np.divide((im * mask).sum(axis=1), mask.sum(axis=1),
                             out=np.zeros(im.shape[0]), where=valid_rows)
plt.figure(figsize=(7, 3))
plt.plot(prof(raw.astype(float)), label="raw")
plt.plot(prof(raw_dgc.astype(float)), label="depth-gain normalised")
plt.xlabel("image row (depth)"); plt.ylabel("mean intensity in fan")
plt.title("Row-mean depth profile", fontsize=11)
plt.legend(); plt.tight_layout(); plt.show()

## 5. Stage 2 — Despeckle filtering

All five filters share a common sliding-neighbourhood principle: a local window is
centred on each pixel and a new value is computed from the pixels it contains. The
methods differ only in how that value is derived.

Parameters below are exactly those documented in the report (Table 4.2) and are held
fixed for every experiment.

### 5.1 Median filter

The median filter replaces each pixel with the centre of its sorted neighbourhood,

$$ I_{\text{out}}(x,y) = \operatorname{median}\{\, I(i,j) : (i,j) \in \mathcal{N}_k(x,y) \,\} $$

It is an order-statistic operator, so it removes isolated bright/dark speckle spikes
without the blurring a mean filter would cause. It has no model of speckle
statistics, however, so edges are eroded once the window is large.

**Parameter:** $k = 9$.

In [ ]:
@despeckle("median")
def f_median(img, k=9):
    return median_filter(img, k)


outputs["median"] = post_process(raw, mask=mask, method="median")
show_pair(raw, outputs["median"], "AFTER - Median (k=9)")

### 5.2 Lee filter

The Lee filter is a local linear minimum-mean-square-error estimator for
multiplicative noise. With local mean $\mu$, local coefficient of variation $C_i$
and speckle coefficient of variation $C_u$,

$$ I_{\text{out}} = \mu + W\,(I - \mu), \qquad
W = 1 - \frac{C_u^{2}}{C_i^{2}} $$

In homogeneous tissue $C_i \approx C_u$, so $W \to 0$ and the output tends to the
local mean (strong smoothing). Near an edge $C_i \gg C_u$, so $W \to 1$ and the
pixel is left essentially untouched — this adaptivity is why Lee preserves edges far
better than the median filter.

**Parameters:** window $k = 11$, $C_u = 0.523$ (fully developed speckle).

In [ ]:
@despeckle("lee")
def f_lee(img, size=11, cu=0.523):
    x = img.astype(np.float64)
    mu = uniform_filter(x, size)
    mu2 = uniform_filter(x * x, size)
    var = np.maximum(mu2 - mu * mu, 0)
    ci2 = var / (mu * mu + 1e-8)
    W = np.clip(1.0 - (cu * cu) / (ci2 + 1e-8), 0, 1)
    return mu + W * (x - mu)


outputs["lee"] = post_process(raw, mask=mask, method="lee")
show_pair(raw, outputs["lee"], "AFTER - Lee (k=11)")

### 5.3 SRAD — Speckle Reducing Anisotropic Diffusion

SRAD formulates despeckling as image evolution under a diffusion equation,

$$ \frac{\partial I}{\partial t} = \operatorname{div}\bigl( c(q)\,\nabla I \bigr), $$

where the diffusion coefficient is driven by the **instantaneous coefficient of
variation** $q$ rather than the gradient magnitude used by Perona–Malik:

$$ c(q) = \frac{1}{1 + \dfrac{q^{2}-q_0^{2}}{q_0^{2}\,(1+q_0^{2})}},
\qquad
q^{2} = \frac{\tfrac12 |\nabla I|^{2} - \tfrac{1}{16}(\nabla^{2} I)^{2}}
              {\bigl(1 + \tfrac14 \nabla^{2} I\bigr)^{2}} $$

$q$ is large at edges (little diffusion) and small in homogeneous speckle (strong
diffusion), and the speckle scale function $q_0(t)$ decays with iteration so
smoothing becomes progressively more conservative. Because it is derived for
multiplicative speckle rather than additive noise, SRAD is the reference method of
the despeckling literature.

**Parameters:** 400 iterations, $\Delta t = 0.12$.

In [ ]:
@despeckle("srad")
def f_srad(img, n=400, dt=0.12):
    x = img.astype(np.float64) + 1.0
    for t in range(n):
        q0 = 1.0 / np.sqrt(max(1e-6, 1.0 + t * 0.012)) * 0.9
        N = np.roll(x, -1, 0); S = np.roll(x, 1, 0)
        E = np.roll(x, -1, 1); W = np.roll(x, 1, 1)
        dN, dS, dE, dW = N - x, S - x, E - x, W - x
        g2 = (dN ** 2 + dS ** 2 + dE ** 2 + dW ** 2) / (x * x)
        lap = (dN + dS + dE + dW) / x
        num = 0.5 * g2 - (1 / 16.0) * lap ** 2
        den = (1.0 + 0.25 * lap) ** 2 + 1e-8
        q2 = np.maximum(num / den, 0)
        c = np.clip(1.0 / (1.0 + (q2 - q0 * q0) / (q0 * q0 * (1 + q0 * q0) + 1e-8)), 0, 1)
        cN, cS = c, np.roll(c, 1, 0)
        cE, cW = c, np.roll(c, 1, 1)
        x = x + (dt / 4.0) * (cN * dN + cS * dS + cE * dE + cW * dW)
    return x - 1.0


outputs["srad"] = post_process(raw, mask=mask, method="srad")
show_pair(raw, outputs["srad"], "AFTER - SRAD (400 iterations, dt=0.12)")

### 5.4 Non-Local Means

Non-local means abandons the local neighbourhood entirely. Each pixel is replaced by
a weighted average of *all* pixels whose surrounding **patch** looks similar:

$$ I_{\text{out}}(p) = \frac{1}{Z(p)}\sum_{q \in \Omega} w(p,q)\, I(q),
\qquad
w(p,q) = \exp\!\left(-\frac{\lVert P(p) - P(q) \rVert_2^{2}}{h^{2}}\right) $$

Speckle is uncorrelated between similar patches while true structure is not, so
averaging over many similar patches suppresses speckle strongly. The cost is
computational: every pixel is compared against a whole search window, making NLM the
slowest method evaluated here.

**Parameters:** patch $5 \times 5$, search radius 11, $h = 0.16$.

In [ ]:
@despeckle("nlm")
def f_nlm(img, h=0.16, sigma=0.10, patch_size=5, patch_distance=11):
    x = img.astype(np.float64) / 255.0
    y = denoise_nl_means(x, h=h, sigma=sigma, fast_mode=True,
                          patch_size=patch_size, patch_distance=patch_distance)
    return y * 255.0


outputs["nlm"] = post_process(raw, mask=mask, method="nlm")
show_pair(raw, outputs["nlm"], "AFTER - Non-Local Means (5x5 patch, radius 11)")

### 5.5 Wavelet soft thresholding

Log compression turns multiplicative speckle into an approximately additive
component, so the image is first mapped through $\log(1+I)$. A Daubechies-4
decomposition is then taken and each detail coefficient is soft-thresholded,

$$ \hat{w} = \operatorname{sign}(w)\,\max\bigl(|w| - \lambda,\; 0\bigr), $$

before reconstruction and the inverse exponential mapping. Small coefficients
(dominated by speckle) are removed while large ones (edges) survive, shrunk by
$\lambda$.

**Parameters:** Daubechies-4, 4 decomposition levels, soft thresholding.

In [ ]:
@despeckle("wavelet")
def f_wavelet(img, wavelet="db4", levels=4, sigma=0.10):
    x = np.log1p(img.astype(np.float64) / 255.0)
    y = denoise_wavelet(x, sigma=sigma, wavelet=wavelet, mode="soft",
                         wavelet_levels=levels, rescale_sigma=True)
    return np.expm1(y) * 255.0


outputs["wavelet"] = post_process(raw, mask=mask, method="wavelet")
show_pair(raw, outputs["wavelet"], "AFTER - Wavelet soft threshold (db4, 4 levels)")

## 6. Stage 3 — Contrast and edge restoration

Despeckling necessarily removes some local variance, which leaves the image looking
flat. Stage 3 restores perceived contrast in two steps:

1. **CLAHE** — contrast limited adaptive histogram equalisation redistributes
   intensities tile-by-tile, so deep, attenuated regions regain contrast without the
   noise amplification that global equalisation would cause. The clip limit bounds
   that amplification.
2. **Unsharp masking** — a Gaussian-blurred copy is subtracted from the image,
   $I_{\text{sharp}} = (1+a)I - a\,\mathcal{G}_\sigma * I$, re-emphasising boundaries
   that diffusion has softened.

**Parameters:** clip limit 1.3, tile grid $8 \times 8$; unsharp $\sigma = 2.0$,
amount 1.28 / 0.28.

In [ ]:
def clahe_unsharp(img, clip_limit=1.3, tile=(8, 8), sigma=2.0, amount=1.28, subtract=0.28):
    cl = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile).apply(img)
    blur = cv2.GaussianBlur(cl, (0, 0), sigma)
    return cv2.addWeighted(cl, amount, blur, -subtract, 0)


outputs["srad+clahe"] = post_process(raw, mask=mask, method="srad", enhance=True)
show_pair(outputs["srad"], outputs["srad+clahe"],
          "AFTER - SRAD + CLAHE + unsharp", "BEFORE - SRAD only")

## 7. Stage 4 — Quality metrics

Five metrics are computed. Two are reference-free (measurable on real scans where no
ground truth exists) and three are full-reference (available here because the
testbench provides a speckle-free reference).

**Contrast-to-noise ratio** between a lesion ROI and a background ROI:

$$ \mathrm{CNR} = \frac{\lvert \mu_L - \mu_B \rvert}{\sqrt{\tfrac12(\sigma_L^{2} + \sigma_B^{2})}} $$

**Speckle SNR** in a homogeneous region, $\mathrm{SNR}_{\text{spk}} = \mu_B/\sigma_B$,
which measures how much granularity remains.

**Edge preservation index**, the normalised correlation of the Laplacians of the
processed image and the reference, restricted to the valid fan:

$$ \mathrm{EPI} = \frac{\sum (\Delta I_p - \overline{\Delta I_p})(\Delta I_r - \overline{\Delta I_r})}
{\sqrt{\sum (\Delta I_p - \overline{\Delta I_p})^{2}\sum (\Delta I_r - \overline{\Delta I_r})^{2}}} $$

**SSIM** and **PSNR** against the ground truth complete the set. Higher is better in
every case. EPI for the raw image is 1.000 by definition — the unprocessed image
defines the unfiltered edge content.

In [ ]:
H, W = raw.shape


def disk(cy, cx, rad):
    yy, xx = np.mgrid[0:H, 0:W]
    return ((yy - cy) ** 2 + (xx - cx) ** 2) <= rad * rad


LESION = disk(int(0.36 * H), int(0.40 * W), 28)      # inside the anechoic cyst
BACKGROUND = disk(int(0.36 * H), int(0.66 * W), 28)  # homogeneous tissue


def cnr(x):
    a, b = x[LESION], x[BACKGROUND]
    return abs(a.mean() - b.mean()) / np.sqrt(0.5 * (a.var() + b.var()) + 1e-9)


def snr_speckle(x):
    b = x[BACKGROUND]
    return b.mean() / (b.std() + 1e-9)


def epi(x, ref):
    lx = cv2.Laplacian(x.astype(np.float32), cv2.CV_32F)
    lr = cv2.Laplacian(ref.astype(np.float32), cv2.CV_32F)
    lx = lx[mask] - lx[mask].mean()
    lr = lr[mask] - lr[mask].mean()
    return float(np.sum(lx * lr) / np.sqrt(np.sum(lx * lx) * np.sum(lr * lr) + 1e-12))


fig, ax = plt.subplots(figsize=(6.5, 4))
ax.imshow(raw, vmin=0, vmax=255); ax.axis("off")
ax.add_patch(plt.Circle((0.40 * W, 0.36 * H), 28, fc="none", ec="#C00000", lw=2))
ax.add_patch(plt.Circle((0.66 * W, 0.36 * H), 28, fc="none", ec="#70AD47", lw=2))
ax.set_title("CNR regions: lesion (red) vs. background (green)", fontsize=11)
plt.tight_layout(); plt.show()

## 8. Results

The table below is computed from the images produced in Sections 5 and 6 and is
checked cell-by-cell against Table 3.1 of the project report.

In [ ]:
LABELS = {"median": "Median", "lee": "Lee", "wavelet": "Wavelet",
          "nlm": "NLM", "srad": "SRAD", "srad+clahe": "SRAD + CLAHE"}
gt_f, raw_f = gt.astype(float), raw.astype(float)

rows = {"Original (raw)": {"CNR": cnr(raw_f), "SNR_spk": snr_speckle(raw_f),
                            "EPI": 1.0, "SSIM": ssim(gt_f, raw_f, data_range=255),
                            "PSNR": psnr(gt_f, raw_f, data_range=255)}}
for key in ["median", "lee", "wavelet", "nlm", "srad", "srad+clahe"]:
    x = outputs[key].astype(float)
    rows[LABELS[key]] = {"CNR": cnr(x), "SNR_spk": snr_speckle(x), "EPI": epi(x, gt_f),
                          "SSIM": ssim(gt_f, x, data_range=255),
                          "PSNR": psnr(gt_f, x, data_range=255)}

results = pd.DataFrame(rows).T[["CNR", "SNR_spk", "EPI", "SSIM", "PSNR"]]
results.round(3)

### 8.1 Parity check against the report

Table 3.1 of the report is hard-coded below and compared against the freshly
computed values, to tolerances of half the report's last printed decimal place.

In [ ]:
REPORT_TABLE = {   # report Chapter 3, Table 3.1 (tab:results)
    "Original (raw)": {"CNR": 1.78, "SNR_spk": 4.28, "EPI": 1.000, "SSIM": 0.484},
    "Median":         {"CNR": 2.45, "SNR_spk": 7.07, "EPI": 0.759, "SSIM": 0.625},
    "Lee":            {"CNR": 2.58, "SNR_spk": 7.55, "EPI": 0.943, "SSIM": 0.667},
    "Wavelet":        {"CNR": 2.00, "SNR_spk": 5.08, "EPI": 0.570, "SSIM": 0.509},
    "NLM":            {"CNR": 3.03, "SNR_spk": 9.77, "EPI": 0.806, "SSIM": 0.728},
    "SRAD":           {"CNR": 2.98, "SNR_spk": 9.29, "EPI": 0.955, "SSIM": 0.744},
    "SRAD + CLAHE":   {"CNR": 2.27, "SNR_spk": 5.02, "EPI": 0.827, "SSIM": 0.640},
}
TOL = {"CNR": 0.005, "SNR_spk": 0.005, "EPI": 0.0005, "SSIM": 0.0005}

mismatches = []
for method, expected in REPORT_TABLE.items():
    for col, want in expected.items():
        got = float(results.loc[method, col])
        if abs(got - want) > TOL[col]:
            mismatches.append(f"{method:15s} {col:8s} report={want}  computed={got:.4f}")

print(f"checked {sum(len(v) for v in REPORT_TABLE.values())} values against report Table 3.1")
print("PARITY: PASS - every value matches the report" if not mismatches else "PARITY: MISMATCH")
for m in mismatches:
    print("  ", m)

### 8.2 Metric comparison chart

In [ ]:
order = ["Original (raw)", "Median", "Lee", "Wavelet", "NLM", "SRAD"]
panels = [("CNR", "Contrast (higher = better)", "#2479C6"),
          ("SNR_spk", "Speckle suppression (higher = better)", "#ED7D31"),
          ("EPI", "Edge preservation (higher = better)", "#70AD47")]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (col, title, colour) in zip(axes, panels):
    vals = [results.loc[m, col] for m in order]
    bars = ax.bar(range(len(order)), vals, color=[colour] * len(order))
    bars[0].set_color("#BFBFBF")
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=35, ha="right", fontsize=9)
    ax.set_title(title, fontsize=10.5)
    ax.spines[["top", "right"]].set_visible(False)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v * 1.02, f"{v:.2f}",
                ha="center", fontsize=8.5)
plt.tight_layout(); plt.show()

### 8.3 Side-by-side comparison

In [ ]:
panel = [(raw, "Original (raw)"), (outputs["median"], "Median"), (outputs["lee"], "Lee"),
         (outputs["srad"], "SRAD  (recommended)"), (outputs["nlm"], "Non-Local Means"),
         (outputs["wavelet"], "Wavelet")]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (im, title) in zip(axes.ravel(), panel):
    ax.imshow(im, vmin=0, vmax=255); ax.axis("off")
    ax.set_title(title, fontsize=11,
                 color="#ED7D31" if title.startswith("SRAD") else "black")
plt.tight_layout(); plt.show()

## 9. Demonstration on real clinical ultrasound images

Sections 3–8 use the synthetic testbench, because it is the only way to obtain a
speckle-free reference and therefore the only way to measure PSNR, SSIM and EPI.
This section applies the *same* pipeline to four **real clinical B-mode images** of
different tissue types, to show that the module behaves sensibly on genuine scanner
output rather than only on simulated data.

**Images and licensing.** All four are openly licensed images from Wikimedia
Commons, by Nevit Dilmen, under **CC BY-SA 3.0**. Full attribution — author,
licence, and source page for each — is recorded in `img/real/ATTRIBUTION.json` and
printed by the cell below, and must be reproduced if these figures are used in the
report. Scanner-burned annotations (text overlays, the grey-scale bar, depth ticks)
were cropped away first, since they are not tissue and would corrupt the region
statistics.

**Two honest caveats:**

1. **No ground truth exists for real scans.** There is no speckle-free reference
   image, so **PSNR, SSIM and EPI cannot be computed here** — they are all
   full-reference metrics. Only the two reference-free metrics, CNR and
   $\mathrm{SNR}_{\text{spk}}$, are reported below. Nothing is substituted for the
   missing reference.
2. **These are display-format images, not raw module output.** They have already
   been through the scanner vendor's own post-processing chain, which typically
   includes some proprietary speckle reduction. They therefore match the *interface*
   the module expects (8-bit, log-compressed, scan-converted) but they are not
   "raw" in the sense the upstream B-Mode Image Formation module would deliver.

Consequently this section is a **qualitative demonstration**, not a second
quantitative validation. The measured results of the project remain those of
Section 8.

In [ ]:
import json

REAL_DIR = "img/real"

# ROI pairs were placed by visual inspection of each cropped image
REAL = {
    "liver":   dict(les=(100, 307, 28), bg=(250, 200, 32),
                    note="hyperechoic focal lesion vs. liver parenchyma"),
    "kidney":  dict(les=(185, 270, 22), bg=(350, 250, 32),
                    note="echogenic renal sinus vs. renal cortex"),
    "breast":  dict(les=(72, 228, 28), bg=(250, 200, 35),
                    note="hypoechoic mass vs. fibroglandular tissue"),
    "thyroid": dict(les=(170, 180, 40), bg=(120, 300, 30),
                    note="hypoechoic nodule vs. normal parenchyma"),
}

credits = json.load(open(f"{REAL_DIR}/ATTRIBUTION.json"))
for organ, c in credits.items():
    print(f"{organ:8s} {c['license']:13s} {c['author']:22s} {c['source_page']}")

In [ ]:
def load_real(organ):
    """Load a cropped real scan and derive its valid-tissue mask."""
    im = cv2.imread(f"{REAL_DIR}/{organ}_gray.png", cv2.IMREAD_GRAYSCALE)
    m = (im > 0).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8)).astype(bool)
    return im, m


def roi_mask(shape, cy, cx, rad):
    yy, xx = np.mgrid[0:shape[0], 0:shape[1]]
    return ((yy - cy) ** 2 + (xx - cx) ** 2) <= rad * rad


def cnr_roi(x, L, B):
    a, b = x[L], x[B]
    return abs(a.mean() - b.mean()) / np.sqrt(0.5 * (a.var() + b.var()) + 1e-9)


METHODS = ["median", "lee", "srad", "nlm", "wavelet"]
real_imgs, real_rows = {}, []

for organ, spec in REAL.items():
    im, m = load_real(organ)
    L = roi_mask(im.shape, *spec["les"])
    B = roi_mask(im.shape, *spec["bg"])
    per = {"raw": im}
    for method in METHODS:
        per[method] = post_process(im, mask=m, method=method)
    real_imgs[organ] = (per, L, B)
    for name, img in per.items():
        f = img.astype(float)
        real_rows.append({"tissue": organ, "method": name, "CNR": cnr_roi(f, L, B),
                           "SNR_spk": f[B].mean() / (f[B].std() + 1e-9)})

real_long = pd.DataFrame(real_rows)
order = ["raw"] + METHODS
cnr_tab = real_long.pivot(index="tissue", columns="method", values="CNR")[order]
snr_tab = real_long.pivot(index="tissue", columns="method", values="SNR_spk")[order]

print("CNR on real clinical images (higher = better)")
display(cnr_tab.round(3))
print("\nSpeckle SNR on real clinical images (higher = better)")
display(snr_tab.round(3))
print("\nPSNR / SSIM / EPI: not computable - no speckle-free reference exists for real scans.")

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(11, 17))
for row, (organ, (per, L, B)) in zip(axes, real_imgs.items()):
    spec = REAL[organ]
    row[0].imshow(per["raw"], vmin=0, vmax=255)
    row[0].add_patch(plt.Circle((spec["les"][1], spec["les"][0]), spec["les"][2],
                                 fc="none", ec="#FF3B30", lw=2))
    row[0].add_patch(plt.Circle((spec["bg"][1], spec["bg"][0]), spec["bg"][2],
                                 fc="none", ec="#34C759", lw=2))
    row[0].set_title(f"{organ.capitalize()} - raw clinical scan\n{spec['note']}", fontsize=10)
    row[1].imshow(per["srad"], vmin=0, vmax=255)
    row[1].set_title(f"{organ.capitalize()} - after SRAD\n"
                     f"CNR {cnr_tab.loc[organ, 'raw']:.2f} -> {cnr_tab.loc[organ, 'srad']:.2f}",
                     fontsize=10, color="#ED7D31")
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
per, _, _ = real_imgs["thyroid"]
panel = [("raw", "Raw clinical scan"), ("median", "Median"), ("lee", "Lee"),
         ("srad", "SRAD  (recommended)"), ("nlm", "Non-Local Means"), ("wavelet", "Wavelet")]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (key, title) in zip(axes.ravel(), panel):
    ax.imshow(per[key], vmin=0, vmax=255); ax.axis("off")
    ax.set_title(title, fontsize=11, color="#ED7D31" if key == "srad" else "black")
fig.suptitle("Full filter bank on a real thyroid scan with a hypoechoic nodule",
             fontsize=12.5, y=1.00)
plt.tight_layout()
plt.show()

## 10. Conclusion

Every filter improved both contrast-to-noise ratio and speckle SNR over the raw
image, but the spread is substantial and the ranking depends on which property is
valued.

- **NLM** achieves the highest raw noise performance (CNR 3.03, speckle SNR 9.77)
  but preserves only 0.806 of the edge information and is by a wide margin the
  slowest method evaluated.
- **SRAD** attains essentially the same noise performance — CNR 2.98, a difference
  under 2% — while retaining the highest edge preservation (EPI 0.955) and the
  highest structural similarity to the ground truth (SSIM 0.744), at moderate
  computational cost.
- **Lee** is the best of the fast methods (CNR 2.58, EPI 0.943) and remains a sound
  choice when compute is constrained.
- **Median** suppresses speckle well but erodes edges (EPI 0.759).
- **Wavelet** thresholding is the weakest here on every axis (EPI 0.570).

**SRAD is therefore the recommended default filter for this module** — it gives the
best balance of speckle suppression and edge preservation, which is the property
that matters for lesion conspicuity in a diagnostic image. The optional Stage 3
CLAHE + unsharp pass trades measured CNR and EPI for visual contrast and is offered
as a display option rather than as the default.

Section 9 applies the same pipeline to four real clinical B-mode scans — liver,
kidney, breast and thyroid. SRAD improves the reference-free metrics on genuine
scanner output as well, which supports the choice qualitatively, though those images
carry no ground truth and so cannot extend the quantitative result.

**Limitations and future work.** The measured results are obtained on a synthetic
testbench with known ground truth, not on production output from the upstream B-Mode
Image Formation module, which had not yet been delivered. The real scans of Section 9
are a qualitative check only: they have no speckle-free reference, so PSNR, SSIM and
EPI cannot be computed on them, and they have already passed through a scanner
vendor's own processing chain. The pipeline and its metrics should be re-validated
once real upstream module output is available. The scope is deliberately restricted
to classical image processing; learning-based despeckling is outside the boundary set
for this module.